# Supervised reproduction: Figures 1-3

This notebook reproduces the supervised-learning overview and the feature-importance comparisons from the main article:

> Vallerio, M., del Rio Chanona, A., & Navarro-Brull, F. J. (2026). *All you need is noise - from feature selection to explainable industrial AI*. **Digital Chemical Engineering, 18**, 100290.
> Local article copy: `/Users/b42549592/Documents/databelts/COIQCV/publications/2026-Noise.pdf`.

The paper's supervised section uses the distillation-tower dataset to demonstrate **synthetic noise features** (SNFs). These are variables known in advance to be uninformative. When a model ranks a real process variable below the strongest SNF, the paper treats that variable as indistinguishable from noise for that fitted model.

This notebook is self-contained. It reads the local Excel workbook, leaves missing predictor values as `NaN`, and uses LightGBM only. The reproduced values are expected to be close to, but not exactly identical to, the JMP output in the article.

In [ ]:
# Self-contained runtime setup.
# The notebook installs only the packages it needs when they are missing.
# scikit-learn is intentionally not used; LightGBM can handle missing predictor
# values directly, so rows with NaN predictors are not discarded.
import importlib.util
import subprocess
import sys
import textwrap


def ensure(import_name, package_name=None):
    """Install a Python package only when its import is unavailable."""
    package_name = package_name or import_name
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])


for import_name, package_name in [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("openpyxl", "openpyxl"),
    ("matplotlib", "matplotlib"),
    ("lightgbm", "lightgbm"),
]:
    ensure(import_name, package_name)

try:
    import lightgbm  # noqa: F401
except OSError as exc:
    # On macOS, pip-installed LightGBM may need OpenMP at runtime.
    # Raising an explicit message is clearer than the raw dynamic-library error.
    if "libomp" in str(exc):
        raise RuntimeError(textwrap.dedent('''
        LightGBM is installed, but macOS cannot find `libomp.dylib`.
        Install OpenMP once and rerun the notebook:

            mamba install -c conda-forge llvm-openmp lightgbm

        or:

            brew install libomp
        ''')) from exc
    raise

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb

warnings.filterwarnings("ignore", category=UserWarning)

# Use one plotting style for every reproduced figure.
plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 220,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.22,
    "font.size": 10,
})


# Locate the workbook whether the notebook is run from this folder or elsewhere.
def find_project_root():
    candidates = [
        Path.cwd(),
        Path("/Users/b42549592/Documents/GitHub/all-you-need-is-noise/01_supervised"),
    ]
    for candidate in candidates:
        if (candidate / "distillation_tower_noise_paper.xlsx").exists():
            return candidate
    raise FileNotFoundError("Could not locate distillation_tower_noise_paper.xlsx")


ROOT = find_project_root()
DATA_PATH = ROOT / "distillation_tower_noise_paper.xlsx"
OUTPUT_DIR = ROOT / "results" / "python" / "trees" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load the starting data exactly once. LightGBM receives NaNs unchanged later.
raw = pd.read_excel(DATA_PATH, sheet_name="distillation_tower_noise_paper")
raw = raw.sort_values("Date").reset_index(drop=True)

# Process measurements used as candidate supervised predictors.
sensor_features = [
    "OC1", "Temp11", "Temp12", "PressureC1", "TempC1", "Temp1", "FlowC1",
    "Temp2", "Temp3", "TempC2", "TempC3", "Temp4", "Temp5", "Temp6",
    "Temp7", "Temp8", "FlowC9", "FlowC2", "Temp9", "Temp10", "FlowC3",
    "FlowC4", "VapourPressure",
]
# Synthetic-noise features provide a built-in noise floor for feature importance.
snf_features = ["Shuffle[yield]", "Random Uniform Noise", "Random Normal Noise"]
feature_cols = sensor_features + snf_features

print(f"LightGBM {lgb.__version__}")
print(f"Rows: {len(raw)}")
print(f"Data: {DATA_PATH}")

## Figure 1: supervised-learning dataset

Figure 1 is the orientation figure for the supervised problem. Panel (a) shows each sensor in its own colored lane so temporal structure and missing segments are visible without collapsing all tags onto one axis. Panel (b) shows the response variable, product yield. Panel (c) summarizes the relationship between two selected process variables and yield with hexagonal bins colored by mean yield. Panel (d) puts selected variables and yield on a common scaled axis for a compact multivariate view.

In [ ]:
# Build the four-panel dataset overview used as Figure 1.
fig, axes = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)
date = raw["Date"]

# (a) Normalize each sensor independently, then offset it into its own colored lane.
norm = raw[sensor_features].astype(float)
norm = (norm - norm.min()) / (norm.max() - norm.min())
lane_gap = 1.12
cmap_lanes = plt.get_cmap("turbo")
lane_colors = {col: cmap_lanes(i / max(1, len(sensor_features) - 1)) for i, col in enumerate(sensor_features)}
for lane, col in enumerate(sensor_features):
    offset = lane * lane_gap
    axes[0, 0].plot(date, norm[col] + offset, color=lane_colors[col], lw=0.9, alpha=0.9)
axes[0, 0].set_yticks(np.arange(len(sensor_features)) * lane_gap + 0.5)
axes[0, 0].set_yticklabels(sensor_features, fontsize=7)
axes[0, 0].set_title("(a) Individual sensor lanes")
axes[0, 0].set_ylabel("Sensor tag")
axes[0, 0].grid(axis="x", alpha=0.18)
axes[0, 0].grid(axis="y", alpha=0.08)

# (b) Plot the response over time and add a small inset histogram for its marginal distribution.
axes[0, 1].plot(date, raw["yield"], color="#222222", marker="o", ms=2.2, lw=1)
axes[0, 1].set_title("(b) Product yield")
axes[0, 1].set_ylabel("Yield")
ax_hist = axes[0, 1].inset_axes([0.63, 0.55, 0.32, 0.35])
ax_hist.hist(raw["yield"], bins=18, color="#7AA6C2", edgecolor="white")
ax_hist.set_xticks([])
ax_hist.set_yticks([])
ax_hist.set_title("distribution", fontsize=8)

# (c) Aggregate points into hexagons and color each bin by the mean yield in that bin.
hb = axes[1, 0].hexbin(
    raw["FlowC1"], raw["Temp1"], C=raw["yield"],
    reduce_C_function=np.mean, gridsize=18, cmap="viridis",
    mincnt=1, linewidths=0.25, edgecolors="white",
)
axes[1, 0].set_title("(c) Mean yield over FlowC1 and Temp1")
axes[1, 0].set_xlabel("FlowC1")
axes[1, 0].set_ylabel("Temp1")
fig.colorbar(hb, ax=axes[1, 0], label="Mean yield")

# (d) parallel coordinates including yield
parallel_cols = ["FlowC1", "Temp1", "Delta[PressureC1]", "yield"]
parallel = raw[parallel_cols].dropna().iloc[::3].copy()
scaled = parallel[parallel_cols].copy()
scaled = (scaled - scaled.min()) / (scaled.max() - scaled.min())
x = np.arange(len(parallel_cols))
colors = plt.get_cmap("viridis")(
    (parallel["yield"] - parallel["yield"].min()) / (parallel["yield"].max() - parallel["yield"].min())
)
for i, row in scaled.iterrows():
    axes[1, 1].plot(x, row.to_numpy(), color=colors[scaled.index.get_loc(i)], alpha=0.45, lw=1)
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(parallel_cols)
axes[1, 1].set_ylabel("Min-max scaled value")
axes[1, 1].set_title("(d) Parallel coordinates")

fig.suptitle("Figure 1. Distillation-column data used for supervised learning", fontsize=14, y=1.02)
fig.savefig(OUTPUT_DIR / "figure_1_dataset_overview.png", bbox_inches="tight")
plt.show()

**Figure 1 caption.** Supervised distillation-tower dataset used in the article. (a) Individual colored lanes show the available sensor trajectories over time. (b) Product yield is the supervised target. (c) Hexagonal bins show mean yield over the `FlowC1` and `Temp1` operating region. (d) A scaled multivariate view includes yield alongside selected process and synthetic-noise variables.

## Figures 2 and 3: model comparison with synthetic-noise features

Figures 2 and 3 compare the same three model families used in the supervised section: a 100-tree random forest analogue, a 144-tree boosted tree, and a single decision tree. For Figure 2 the target is the real yield. For Figure 3 the target is the shuffled yield, which should destroy the real predictor-response relationship.

The red bars are synthetic-noise features. They are not removed from the model; they are included intentionally so their importance can be used as a practical threshold.

In [ ]:
# Small NumPy implementation avoids pulling in scikit-learn for one metric.
def r2_score_np(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[mask]
    y_pred = y_pred[mask]
    return float(1 - np.sum((y_true - y_pred) ** 2) / np.sum((y_true - y_true.mean()) ** 2))


# Keep rows where the target exists; predictor NaNs remain in X for LightGBM.
def make_xy(target_col, features):
    y_series = raw[target_col].astype(float)
    keep = y_series.notna()
    X = raw.loc[keep, features].astype(float)
    y = y_series.loc[keep].to_numpy(dtype=float)
    return X, y


# Train one LightGBM model. Feature names are replaced with safe tokens because
# LightGBM rejects some characters used in the original process tags.
def train_lgb(X, y, params, seed):
    safe_names = [f"f{i:02d}" for i in range(X.shape[1])]
    dataset = lgb.Dataset(X.to_numpy(), label=y, feature_name=safe_names, free_raw_data=False)
    train_params = {
        "objective": "regression",
        "metric": "rmse",
        "verbosity": -1,
        "seed": seed,
        "feature_fraction_seed": seed,
        "bagging_seed": seed,
        "data_random_seed": seed,
        "deterministic": True,
        "force_col_wise": True,
        **params,
    }
    rounds = int(train_params.pop("n_estimators"))
    return lgb.train(train_params, dataset, num_boost_round=rounds)


# Model settings approximate the article/JMP comparisons.
# JMP random forest default is represented as 100 trees; the boosted and single-tree
# cases use LightGBM GBDT settings with the requested tree counts.
model_configs = {
    "Random forest, 100 trees": {
        "params": {
            "boosting_type": "rf", "n_estimators": 100, "learning_rate": 1.0,
            "bagging_fraction": 0.8, "bagging_freq": 1, "feature_fraction": 1.0,
            "num_leaves": 64, "min_data_in_leaf": 5,
        },
        "seed": 2026,
    },
    "Boosted tree, 144 trees": {
        "params": {
            "boosting_type": "gbdt", "n_estimators": 144, "learning_rate": 0.1,
            "bagging_fraction": 1.0, "bagging_freq": 0, "feature_fraction": 1.0,
            "num_leaves": 3, "min_data_in_leaf": 5,
        },
        "seed": 2026,
    },
    "Single tree, one estimator": {
        "params": {
            "boosting_type": "gbdt", "n_estimators": 1, "learning_rate": 1.0,
            "bagging_fraction": 1.0, "bagging_freq": 0, "feature_fraction": 0.95,
            "num_leaves": 39, "min_data_in_leaf": 1,
        },
        "seed": 10,
    },
}


# Convert LightGBM gain and split counts back to the original feature names.
def importance_table(model, model_name, target_name):
    gain = model.feature_importance(importance_type="gain").astype(float)
    split = model.feature_importance(importance_type="split").astype(float)
    total = gain.sum()
    return pd.DataFrame({
        "target": target_name,
        "model": model_name,
        "Term": feature_cols,
        "gain_portion": np.where(total > 0, gain / total, 0),
        "split_count": split,
        "is_snf": [name in snf_features for name in feature_cols],
    }).sort_values("gain_portion", ascending=False).reset_index(drop=True)


# Fit the same model configurations once to the real target and once to the shuffled target.
targets = {"yield": "yield", "shuffled yield": "yield (random shuffled)"}
metrics = []
importances = []
for target_name, target_col in targets.items():
    X, y = make_xy(target_col, feature_cols)
    for model_name, config in model_configs.items():
        model = train_lgb(X, y, config["params"], config["seed"])
        pred = model.predict(X.to_numpy())
        metrics.append({
            "target": target_name,
            "model": model_name,
            "training_R2": r2_score_np(y, pred),
        })
        importances.append(importance_table(model, model_name, target_name))

metrics = pd.DataFrame(metrics)
importances = pd.concat(importances, ignore_index=True)
metrics.to_csv(OUTPUT_DIR / "figures_1_to_3_metrics.csv", index=False)
importances.to_csv(OUTPUT_DIR / "figures_1_to_3_importance.csv", index=False)
display(metrics)

In [ ]:
# Draw one row of feature-importance panels for a target.
# Each panel uses the same sorted gain-portion metric and colors SNFs red.
def plot_importance_panels(target_name, title, filename):
    fig, axes = plt.subplots(1, 3, figsize=(15.5, 5.4), constrained_layout=True)
    for ax, model_name in zip(axes, model_configs):
        data = importances[
            (importances["target"] == target_name) &
            (importances["model"] == model_name)
        ].head(14).iloc[::-1]
        colors = np.where(data["is_snf"], "#C83E35", "#3E75A8")
        ax.barh(data["Term"], data["gain_portion"], color=colors)
        ax.set_title(model_name)
        ax.set_xlabel("LightGBM gain portion")
        ax.grid(axis="x", alpha=0.25)
    fig.suptitle(title, fontsize=14)
    fig.savefig(OUTPUT_DIR / filename, bbox_inches="tight")
    plt.show()


plot_importance_panels(
    "yield",
    "Figure 2. Product-yield feature importance with synthetic-noise threshold",
    "figure_2_real_yield_importance.png",
)

**Figure 2 caption.** Feature-importance comparison for the real product-yield target. Blue bars are process variables and red bars are synthetic-noise features. Variables below the strongest synthetic-noise feature are interpreted as below the model-specific noise threshold.

In [ ]:
plot_importance_panels(
    "shuffled yield",
    "Figure 3. Shuffled-yield feature importance with synthetic-noise threshold",
    "figure_3_shuffled_yield_importance.png",
)

**Figure 3 caption.** Feature-importance comparison after replacing the target with shuffled yield. Because the target-response relationship has been broken, the remaining importance patterns show how each model can still assign apparent importance under a no-signal target.